In [12]:
import os
import platform
import pandas as pd
import numpy as np
import torch
import pytorch_lightning as pl

from pytorch_lightning.callbacks.early_stopping import EarlyStopping
from torch.utils.data import DataLoader
from multiprocessing import cpu_count

from model.dkt import DKTModule
from model.sakt import SAKTModule
from model.dkvmn import DKVMNModule

In [13]:
seed = 42
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
pl.seed_everything(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

Global seed set to 42


In [14]:
# SEQ_LEN = 50

SEQ_LEN = 1317
BATCH_SIZE = 16
# BATCH_SIZE = 2
EMBED_DIM = 128
NUM_WORKERS = 0 if platform.system() == 'Windows' else cpu_count()
# dkt, and sakt, and dkvmn...
model = 'dkt'
# model = 'dkvmn'
# model = 'sakt'
q_is_s = True
print("os:{}, num-workers:{}".format(platform.system(), NUM_WORKERS))

os:Linux, num-workers:16


In [15]:
df_train = pd.read_csv('dataset/algebra_2006_2007/train.csv', low_memory=False, encoding="ISO-8859-1")
df_val = pd.read_csv('dataset/algebra_2006_2007/test.csv', low_memory=False, encoding="ISO-8859-1")
df_train.head()

,user_id,q_idx,s_idx,q_type,q_diff,ms_first_response,correct
0,271qgzl01euj,13359,10,7,0.883450,0.120948,1
1,271qgzl01euj,13359,11,7,0.883450,0.007170,1
2,271qgzl01euj,29143,12,7,0.857143,0.030237,0
3,271qgzl01euj,29143,13,7,0.857143,0.002805,1
4,271qgzl01euj,29144,12,7,1.000000,0.003741,1


In [16]:
key_q = 'q_idx'
key_s = 's_idx'
key_qtype = 'q_type'

In [17]:
N_QUESTION = len(df_train[key_q].unique()) + len(df_val[key_q].unique())
N_SKILL = len(df_train[key_s].unique()) + len(df_val[key_s].unique())
N_QUESTION_TYPE = len(df_train[key_qtype].unique()) + len(df_val[key_qtype].unique())
N_QUESTION, N_SKILL, N_QUESTION_TYPE

(99787, 2556, 304)

In [18]:
# 我们需要将数据进行预处理，每个学生的学习记录利用group by合并为序列。
KEY = key_s if q_is_s else key_q
NUM_Q_OR_S = N_SKILL if q_is_s else N_QUESTION

In [19]:
def generate_group_by_df(df):
    group = df[['user_id', KEY, 'correct']].groupby(['user_id']).apply(lambda r: (
            r[KEY].values,
            r['correct'].values
            ))
    return group

train, val = generate_group_by_df(df_train), generate_group_by_df(df_val)


In [20]:
from data_loader.dktdataset import DKTDataset

train_dataset = DKTDataset(train, NUM_Q_OR_S, SEQ_LEN)
train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)

val_dataset = DKTDataset(val, NUM_Q_OR_S, SEQ_LEN)
val_dataloader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

print("train:{}, test:{}".format(len(train_dataset), len(val_dataset)))

train:1068, test:268


In [21]:
import warnings
warnings.filterwarnings('ignore')
if model == 'dkt':
    model = DKTModule(n_question=NUM_Q_OR_S)
elif model == 'sakt':
    model = SAKTModule(n_question=NUM_Q_OR_S, max_seq=SEQ_LEN, embed_dim=EMBED_DIM)
elif model == 'dkvmn':
    model = DKVMNModule(n_question=NUM_Q_OR_S)

print("num of question:{}, num of skill:{}".format(N_QUESTION, N_SKILL))
print("question is skill：{}, num_q_or_s:{}".format(q_is_s, NUM_Q_OR_S))
print("model:{}".format(model))

num of question:99787, num of skill:2556
question is skill：True, num_q_or_s:2556
model:DKTModule(
  (loss): BCEWithLogitsLoss()
  (dkt): DKT(
    (embedding): Embedding(5113, 128)
    (lstm): LSTM(128, 256, num_layers=2, batch_first=True, dropout=0.2)
    (pred): Linear(in_features=256, out_features=2556, bias=True)
  )
)


In [22]:
checkpoint_callback = pl.callbacks.ModelCheckpoint(save_top_k=1, verbose=True, monitor='v_auc', mode='max')

# sakt.train_dataloader
trainer = pl.Trainer(
    gpus=1, 
    max_epochs=200, 
    auto_lr_find=True, 
    callbacks=[checkpoint_callback, EarlyStopping(monitor="v_auc", mode="max", patience=6)]
)

trainer.fit(model=model, train_dataloaders=train_dataloader,val_dataloaders=val_dataloader)

GPU available: True, used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name | Type              | Params
-------------------------------------------
0 | loss | BCEWithLogitsLoss | 0     
1 | dkt  | DKT               | 2.2 M 
-------------------------------------------
2.2 M     Trainable params
0         Non-trainable params
2.2 M     Total params
8.932     Total estimated model params size (MB)


Sanity Checking: 0it [00:00, ?it/s]

Training: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

Epoch 0, global step 67: 'v_auc' reached 0.75171 (best 0.75171), saving model to '/home/czy/KT/BRIKT/lightning_logs/version_107/checkpoints/epoch=0-step=67.ckpt' as top 1


...

Validation: 0it [00:00, ?it/s]

Epoch 16, global step 1139: 'v_auc' was not in top 1


Validation: 0it [00:00, ?it/s]

Epoch 17, global step 1206: 'v_auc' was not in top 1


: 